Initialise ifcopenshell libraries and conditions

In [68]:
import numpy as np
import ifcopenshell

# for some reason ifcopenshell.api is not imported when importing ifcopenshell, so we need to import it explicitly
import ifcopenshell.api.root
import ifcopenshell.api.unit
import ifcopenshell.api.context
import ifcopenshell.api.project
import ifcopenshell.api.geometry

model = ifcopenshell.file()

# create the root entity (IfcProject) and set it as the root of the model
ifcopenshell.api.root.create_entity(model, ifc_class="IfcProject")

# set coordinates to be in meters
length = ifcopenshell.api.unit.add_si_unit(model, unit_type="LENGTHUNIT")
ifcopenshell.api.unit.assign_unit(model, units=[length])

# create contexts for 3D model, plan view and body
model3d = ifcopenshell.api.context.add_context(model, context_type="Model")
plan = ifcopenshell.api.context.add_context(model, context_type="Plan")
body = ifcopenshell.api.context.add_context(model, 
    context_type="Model", context_identifier="Body", target_view="MODEL_VIEW", parent=model3d)

# add some contexts for axes and annotations
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Axis", target_view="GRAPH_VIEW", parent=model3d)
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Axis", target_view="GRAPH_VIEW", parent=plan)
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Box", target_view="MODEL_VIEW", parent=model3d)
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Annotation", target_view="PLAN_VIEW", parent=plan)


# create the project structure components for use in IfcRelAggregates
project = model.by_type("IfcProject")[0]
# site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="Main Site")
# building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building")
site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="Main Site")
building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building")
storey = model.create_entity("IfcBuildingStorey", Name="Ground", Elevation=0.0)

# IfcRelAggregates is used to define the hierarchical structure of the building model.
model.create_entity("IfcRelAggregates", RelatingObject=project, RelatedObjects=[site])
model.create_entity("IfcRelAggregates", RelatingObject=site, RelatedObjects=[building])
model.create_entity("IfcRelAggregates", RelatingObject=building, RelatedObjects=[storey])


#23=IfcRelAggregates($,$,$,$,#19,(#20))

Create function to create IfcSpaces

In [69]:
# define a function to add ifcspaces to the model
import profile


def add_space(length, width, name, origin, height = 3.0):
    space = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSpace", name=name)

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin
    ifcopenshell.api.geometry.edit_object_placement(model, product=space, matrix=matrix, is_si=False)

    profile = model.create_entity("IfcRectangleProfileDef",
        ProfileName=f"{name} footprint", ProfileType="AREA", 
        XDim=length, YDim=width)
    
    space_rep = ifcopenshell.api.geometry.add_profile_representation(
        model, context=body, profile=profile, depth=height)
    
    ifcopenshell.api.geometry.assign_representation(
        model, product=space, representation=space_rep)
    
    model.create_entity("IfcRelContainedInSpatialStructure", RelatingStructure=storey, RelatedElements=[space])
    return space
    

# define a function to add ifcwall to the model
def add_wall(length, name: str, origin, rotate):
    wall = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall", name=name)
    
    wall_rep = ifcopenshell.api.geometry.add_wall_representation(model,
        context=body, length=length, height=3.0, thickness=0.2)
    
    matrix = np.eye(4)
    if rotate:
        matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
    matrix[:,3][0:3] = origin
    
    ifcopenshell.api.geometry.edit_object_placement(model, product=wall, matrix=matrix, is_si=False)
    ifcopenshell.api.geometry.assign_representation(model, product=wall, representation=wall_rep)

    # this part is important because it relates the wall back to the hierarchy of the ifc model defined before
    ifcopenshell.api.spatial.assign_container(model, relating_structure=storey, products=[wall])

    return wall

def add_slab(length, width, name, origin):
    slab = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSlab", name=name)

    # Create a rectangular profile for the slab with your desired dimensions
    profile = model.create_entity("IfcRectangleProfileDef", ProfileName="slab", ProfileType="AREA",
        XDim=length, YDim=width)

    slab_rep = ifcopenshell.api.geometry.add_profile_representation(model, context=body, profile=profile, depth=0.2)

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin
    ifcopenshell.api.geometry.edit_object_placement(model, product=slab, matrix=matrix, is_si=True)
    ifcopenshell.api.geometry.assign_representation(model, product=slab, representation=slab_rep)
    ifcopenshell.api.spatial.assign_container(model, relating_structure=storey, products=[slab])
    
# # define a function to add a room to the model, which is just an ifcspace with physical geometry assigned
# def add_room(length, width, name, origin):
#     north = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
#     east = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
#     south = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
#     west = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

#     length_rep = ifcopenshell.api.geometry.add_wall_representation(model,
#         context=body, length=length, height=3.0, thickness=0.2)
#     width_rep = ifcopenshell.api.geometry.add_wall_representation(model,
#         context=body, length=width, height=3.0, thickness=0.2)
    
#     # set north and south walls to be along the length of the room
#     matrix = np.eye(4)
#     matrix[:,3][0:3] = origin

#     ifcopenshell.api.geometry.edit_object_placement(model, product=north, matrix=matrix)


#     room = 1

#     return room


    # define a function to add a room to the model, which is just an ifcspace with physical geometry assigned
def add_room(length, width, name, origin):
    
    room_dict = {}

    room_dict[name+"wallN"] = add_wall(length=length, name=name+"wallN", origin=origin+np.array([0, width, 0]), rotate=False)
    room_dict[name+"wallS"] = add_wall(length=length, name=name+"wallS", origin=origin+np.array([0,-0.2,0]), rotate=False)
    
    room_dict[name+"wallE"] = add_wall(length=width+0.4, name=name+"wallE", origin=origin+np.array([length+0.2, -0.2, 0]), rotate=True)
    room_dict[name+"wallW"] = add_wall(length=width+0.4, name=name+"wallW", origin=origin+np.array([0, -0.2, 0]), rotate=True)

    room_dict[name+"slab"] = add_slab(length=length+0.4, width=width+0.4, name=name+"slab", origin=origin+np.array([length/2, width/2, -0.2]))   
    
    add_space(length=length, width=width, name=name, origin=origin+np.array([length/2, width/2, 0]))

    return [room_dict]

In [70]:
a = add_room(length=6, width=7, name="Room1", origin=(0,0,0))
print(a)

# add openings in wall, then add space boundaries for openings connecting rooms

model.write("Function_Model.ifc")

[{'Room1wallN': #24=IfcWall('3IVDsFhmz4sOG6uyUkDxfk',$,'Room1wallN',$,$,#46,#40,$,$), 'Room1wallS': #47=IfcWall('2jI8hFii91rRhChsP512KK',$,'Room1wallS',$,$,#68,#63,$,$), 'Room1wallE': #69=IfcWall('3DEhiNXtj4Ixh05EsvlRWl',$,'Room1wallE',$,$,#90,#85,$,$), 'Room1wallW': #91=IfcWall('2gQDE93PXABQe8dYtnV61J',$,'Room1wallW',$,$,#112,#107,$,$), 'Room1slab': None}]
